In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/opt/spark-3.3.0-bin-hadoop3"
os.environ["PYSPARK_SUBMIT_ARGS"] = \
"--packages org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.4.2 pyspark-shell"
import findspark
findspark.init('/opt/spark-3.3.0-bin-hadoop3')
from pyspark.sql import SparkSession
os.makedirs("/home/tavares/warehouse", exist_ok=True)
spark = SparkSession.builder \
    .appName("EcommerceDataLake") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hadoop_catalog.type", "hadoop") \
    .config("spark.sql.catalog.hadoop_catalog.warehouse", "file:///home/tavares/warehouse") \
    .config("spark.sql.default.catalog", "hadoop_catalog") \
    .getOrCreate()

spark

In [4]:
spark.sql("""
CREATE TABLE hadoop_catalog.default.vendas_ecommerce (
    venda_id INT,
    produto_nome STRING,
    categoria STRING,
    quantidade INT,
    preco_unitario DOUBLE,
    data_venda DATE,
    cliente_id STRING,
    vendedor_id INT
)
USING iceberg
PARTITIONED BY (year(data_venda), categoria)
""")

DataFrame[]

In [5]:
INSERT INTO hadoop_catalog.default.vendas_ecommerce VALUES
(1, 'Notebook Dell', 'Eletrônicos', 2, 2500.00, DATE('2023-01-15'), 'CLI001', 101),
(2, 'Mouse Logitech', 'Eletrônicos', 5, 80.00, DATE('2023-01-16'), 'CLI002', 102),
(3, 'Mesa Escritório', 'Móveis', 1, 800.00, DATE('2023-02-10'), 'CLI003', 101),
(4, 'Cadeira Gamer', 'Móveis', 2, 600.00, DATE('2023-02-15'), 'CLI001', 103),
(5, 'Smartphone Samsung', 'Eletrônicos', 1, 1200.00, DATE('2023-03-20'), 'CLI004', 102)

SyntaxError: invalid syntax (1882058726.py, line 1)

In [6]:
spark.sql("""
INSERT INTO hadoop_catalog.default.vendas_ecommerce VALUES
(1, 'Notebook Dell', 'Eletrônicos', 2, 2500.00, DATE('2023-01-15'), 'CLI001', 101),
(2, 'Mouse Logitech', 'Eletrônicos', 5, 80.00, DATE('2023-01-16'), 'CLI002', 102),
(3, 'Mesa Escritório', 'Móveis', 1, 800.00, DATE('2023-02-10'), 'CLI003', 101),
(4, 'Cadeira Gamer', 'Móveis', 2, 600.00, DATE('2023-02-15'), 'CLI001', 103),
(5, 'Smartphone Samsung', 'Eletrônicos', 1, 1200.00, DATE('2023-03-20'), 'CLI004', 102)
""")

DataFrame[]

In [7]:
INSERT INTO hadoop_catalog.default.vendas_ecommerce VALUES
(6, 'Tablet iPad', 'Eletrônicos', 1, 3000.00, DATE('2024-01-10'), 'CLI002', 101),
(7, 'Sofá 3 Lugares', 'Móveis', 1, 1500.00, DATE('2024-01-20'), 'CLI005', 103),
(8, 'Monitor 4K', 'Eletrônicos', 2, 800.00, DATE('2024-02-05'), 'CLI003', 102)

SyntaxError: invalid syntax (3564747552.py, line 1)

In [8]:
spark.sql("""
INSERT INTO hadoop_catalog.default.vendas_ecommerce VALUES
(6, 'Tablet iPad', 'Eletrônicos', 1, 3000.00, CAST('2024-01-10' AS DATE), 'CLI002', 101),
(7, 'Sofá 3 Lugares', 'Móveis', 1, 1500.00, CAST('2024-01-20' AS DATE), 'CLI005', 103),
(8, 'Monitor 4K', 'Eletrônicos', 2, 800.00, CAST('2024-02-05' AS DATE), 'CLI003', 102)
""")

DataFrame[]

In [9]:
spark.sql("""
SELECT * 
FROM hadoop_catalog.default.vendas_ecommerce.snapshots
""").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                                      |summary                                                                                                                        

In [10]:
spark.sql("""
SELECT COUNT(*) AS total_snapshots
FROM hadoop_catalog.default.vendas_ecommerce.snapshots
""").show()

+---------------+
|total_snapshots|
+---------------+
|              2|
+---------------+



In [11]:
spark.sql("""
SELECT *
FROM hadoop_catalog.default.vendas_ecommerce.history
ORDER BY made_current_at
""").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-03-24 23:54:17.852|6442142642168008892|null               |true               |
|2026-03-24 23:56:34.896|3135783010680599706|6442142642168008892|true               |
+-----------------------+-------------------+-------------------+-------------------+



In [12]:
spark.sql("""
SELECT snapshot_id, committed_at
FROM hadoop_catalog.default.vendas_ecommerce.snapshots
ORDER BY committed_at
""").show(truncate=False)

+-------------------+-----------------------+
|snapshot_id        |committed_at           |
+-------------------+-----------------------+
|6442142642168008892|2026-03-24 23:54:17.852|
|3135783010680599706|2026-03-24 23:56:34.896|
+-------------------+-----------------------+



In [13]:
spark.sql("""
SELECT *
FROM hadoop_catalog.default.vendas_ecommerce
VERSION AS OF 123456789
WHERE year(data_venda) = 2023
""").show()

IllegalArgumentException: Cannot find snapshot with ID 123456789

In [15]:
snapshot_id = spark.sql("""
SELECT snapshot_id
FROM hadoop_catalog.default.vendas_ecommerce.snapshots
ORDER BY committed_at
LIMIT 1
""").collect()[0][0]

print(snapshot_id)

6442142642168008892


In [16]:
df_atual = spark.sql("""
SELECT *
FROM hadoop_catalog.default.vendas_ecommerce
WHERE data_venda BETWEEN CAST('2023-01-01' AS DATE)
                     AND CAST('2023-12-31' AS DATE)
""")

df_atual.show()

+--------+------------------+-----------+----------+--------------+----------+----------+-----------+
|venda_id|      produto_nome|  categoria|quantidade|preco_unitario|data_venda|cliente_id|vendedor_id|
+--------+------------------+-----------+----------+--------------+----------+----------+-----------+
|       1|     Notebook Dell|Eletrônicos|         2|        2500.0|2023-01-15|    CLI001|        101|
|       2|    Mouse Logitech|Eletrônicos|         5|          80.0|2023-01-16|    CLI002|        102|
|       5|Smartphone Samsung|Eletrônicos|         1|        1200.0|2023-03-20|    CLI004|        102|
|       3|   Mesa Escritório|     Móveis|         1|         800.0|2023-02-10|    CLI003|        101|
|       4|     Cadeira Gamer|     Móveis|         2|         600.0|2023-02-15|    CLI001|        103|
+--------+------------------+-----------+----------+--------------+----------+----------+-----------+



In [17]:
print("Snapshot:", df_snapshot.count())
print("Atual:", df_atual.count())

NameError: name 'df_snapshot' is not defined

In [18]:
snapshot_id = spark.sql("""
SELECT snapshot_id
FROM hadoop_catalog.default.vendas_ecommerce.snapshots
ORDER BY committed_at
LIMIT 1
""").collect()[0][0]

df_snapshot = spark.sql(f"""
SELECT *
FROM hadoop_catalog.default.vendas_ecommerce
VERSION AS OF {snapshot_id}
WHERE data_venda BETWEEN CAST('2023-01-01' AS DATE)
                     AND CAST('2023-12-31' AS DATE)
""")

In [19]:
spark.sql("""
ALTER TABLE hadoop_catalog.default.vendas_ecommerce
ADD COLUMNS (
    desconto DOUBLE,
    canal_venda STRING
)
""")

DataFrame[]

In [20]:
spark.sql("""
DESCRIBE hadoop_catalog.default.vendas_ecommerce
""").show(truncate=False)

+--------------+-----------------+-------+
|col_name      |data_type        |comment|
+--------------+-----------------+-------+
|venda_id      |int              |       |
|produto_nome  |string           |       |
|categoria     |string           |       |
|quantidade    |int              |       |
|preco_unitario|double           |       |
|data_venda    |date             |       |
|cliente_id    |string           |       |
|vendedor_id   |int              |       |
|desconto      |double           |       |
|canal_venda   |string           |       |
|              |                 |       |
|# Partitioning|                 |       |
|Part 0        |years(data_venda)|       |
|Part 1        |categoria        |       |
+--------------+-----------------+-------+



In [21]:
INSERT INTO hadoop_catalog.default.vendas_ecommerce VALUES
(9, 'Headset Gamer', 'Eletrônicos', 3, 250.00, DATE('2024-03-15'), 'CLI006', 101, 10.0, 'online'),
(10, 'Mesa Centro', 'Móveis', 1, 400.00, DATE('2024-03-20'), 'CLI007', 102, 5.0, 'loja_fisica')

SyntaxError: invalid syntax (3974080224.py, line 1)

In [22]:
spark.sql("""
INSERT INTO hadoop_catalog.default.vendas_ecommerce (
    venda_id, produto_nome, categoria, quantidade, preco_unitario,
    data_venda, cliente_id, vendedor_id, desconto, canal_venda
)
VALUES
(9, 'Headset Gamer', 'Eletrônicos', 3, 250.00, CAST('2024-03-15' AS DATE), 'CLI006', 101, 10.0, 'online'),
(10, 'Mesa Centro', 'Móveis', 1, 400.00, CAST('2024-03-20' AS DATE), 'CLI007', 102, 5.0, 'loja_fisica')
""")

DataFrame[]

In [23]:
spark.sql("""
SELECT *
FROM hadoop_catalog.default.vendas_ecommerce
ORDER BY venda_id DESC
""").show()

+--------+------------------+-----------+----------+--------------+----------+----------+-----------+--------+-----------+
|venda_id|      produto_nome|  categoria|quantidade|preco_unitario|data_venda|cliente_id|vendedor_id|desconto|canal_venda|
+--------+------------------+-----------+----------+--------------+----------+----------+-----------+--------+-----------+
|      10|       Mesa Centro|     Móveis|         1|         400.0|2024-03-20|    CLI007|        102|     5.0|loja_fisica|
|       9|     Headset Gamer|Eletrônicos|         3|         250.0|2024-03-15|    CLI006|        101|    10.0|     online|
|       8|        Monitor 4K|Eletrônicos|         2|         800.0|2024-02-05|    CLI003|        102|    null|       null|
|       7|    Sofá 3 Lugares|     Móveis|         1|        1500.0|2024-01-20|    CLI005|        103|    null|       null|
|       6|       Tablet iPad|Eletrônicos|         1|        3000.0|2024-01-10|    CLI002|        101|    null|       null|
|       5|Smartp

In [24]:
spark.sql("""
SELECT 
    venda_id,
    produto_nome,
    desconto,
    canal_venda
FROM hadoop_catalog.default.vendas_ecommerce
WHERE venda_id <= 5
""").show()

+--------+------------------+--------+-----------+
|venda_id|      produto_nome|desconto|canal_venda|
+--------+------------------+--------+-----------+
|       1|     Notebook Dell|    null|       null|
|       2|    Mouse Logitech|    null|       null|
|       5|Smartphone Samsung|    null|       null|
|       3|   Mesa Escritório|    null|       null|
|       4|     Cadeira Gamer|    null|       null|
+--------+------------------+--------+-----------+



In [25]:
spark.sql("""
SELECT 
    venda_id,
    produto_nome,
    desconto,
    canal_venda
FROM hadoop_catalog.default.vendas_ecommerce
WHERE venda_id >= 9
""").show()

+--------+-------------+--------+-----------+
|venda_id| produto_nome|desconto|canal_venda|
+--------+-------------+--------+-----------+
|       9|Headset Gamer|    10.0|     online|
|      10|  Mesa Centro|     5.0|loja_fisica|
+--------+-------------+--------+-----------+



In [26]:
spark.sql("""
UPDATE hadoop_catalog.default.vendas_ecommerce 
SET preco_unitario = preco_unitario * 100 
WHERE categoria = 'Eletrônicos'
""")

DataFrame[]

In [27]:
spark.sql("""
SELECT venda_id, produto_nome, preco_unitario
FROM hadoop_catalog.default.vendas_ecommerce
WHERE categoria = 'Eletrônicos'
""").show()

+--------+------------------+--------------+
|venda_id|      produto_nome|preco_unitario|
+--------+------------------+--------------+
|       6|       Tablet iPad|      300000.0|
|       8|        Monitor 4K|       80000.0|
|       9|     Headset Gamer|       25000.0|
|       1|     Notebook Dell|      250000.0|
|       2|    Mouse Logitech|        8000.0|
|       5|Smartphone Samsung|      120000.0|
+--------+------------------+--------------+



In [28]:
spark.sql("""
SELECT snapshot_id, operation, committed_at
FROM hadoop_catalog.default.vendas_ecommerce.snapshots
ORDER BY committed_at DESC
""").show(truncate=False)

+-------------------+---------+-----------------------+
|snapshot_id        |operation|committed_at           |
+-------------------+---------+-----------------------+
|1990333489359147699|overwrite|2026-03-25 00:50:15.318|
|9186976170233357367|append   |2026-03-25 00:44:18.017|
|3135783010680599706|append   |2026-03-24 23:56:34.896|
|6442142642168008892|append   |2026-03-24 23:54:17.852|
+-------------------+---------+-----------------------+



In [29]:
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vendas_updates AS
SELECT 1 as venda_id, 'Notebook Dell UPDATED' as produto_nome, 'Eletrônicos' as categoria, 
       2 as quantidade, 2600.00 as preco_unitario, CAST('2023-01-15' AS DATE) as data_venda, 
       'CLI001' as cliente_id, 101 as vendedor_id, 0.0 as desconto, 'online' as canal_venda
UNION ALL
SELECT 11 as venda_id, 'Teclado Mecânico' as produto_nome, 'Eletrônicos' as categoria,
       4 as quantidade, 300.00 as preco_unitario, CAST('2024-04-01' AS DATE) as data_venda,
       'CLI008' as cliente_id, 103 as vendedor_id, 15.0 as desconto, 'online' as canal_venda
""")

DataFrame[]

In [30]:
spark.sql("""
MERGE INTO hadoop_catalog.default.vendas_ecommerce AS target
USING vendas_updates AS source
ON target.venda_id = source.venda_id

WHEN MATCHED THEN UPDATE SET
    target.produto_nome = source.produto_nome,
    target.categoria = source.categoria,
    target.quantidade = source.quantidade,
    target.preco_unitario = source.preco_unitario,
    target.data_venda = source.data_venda,
    target.cliente_id = source.cliente_id,
    target.vendedor_id = source.vendedor_id,
    target.desconto = source.desconto,
    target.canal_venda = source.canal_venda

WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[]

In [31]:
spark.sql("""
SELECT venda_id, produto_nome, preco_unitario
FROM hadoop_catalog.default.vendas_ecommerce
WHERE venda_id IN (1, 11)
""").show()

+--------+--------------------+--------------+
|venda_id|        produto_nome|preco_unitario|
+--------+--------------------+--------------+
|      11|    Teclado Mecânico|         300.0|
|       1|Notebook Dell UPD...|        2600.0|
+--------+--------------------+--------------+



In [32]:
spark.sql("""
SELECT COUNT(*) AS total_arquivos
FROM hadoop_catalog.default.vendas_ecommerce.files
""").show()

+--------------+
|total_arquivos|
+--------------+
|             7|
+--------------+



In [33]:
spark.sql("""
SELECT 
    file_path,
    record_count,
    file_size_in_bytes
FROM hadoop_catalog.default.vendas_ecommerce.files
""").show(truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+------------------+
|file_path                                                                                                                                                                |record_count|file_size_in_bytes|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+------------------+
|file:/home/tavares/warehouse/default/vendas_ecommerce/data/data_venda_year=2024/categoria=Eletr%C3%B4nicos/00038-826-bb32ccd9-da7e-48db-a2c4-6a1fd8810cc0-0-00001.parquet|1           |2897              |
|file:/home/tavares/warehouse/default/vendas_ecommerce/data/data_venda_year=2023/categoria=Eletr%C3%B4nicos/00057-827-bb32ccd9-da7e-48db-a2c4-6a1fd8810cc0-0-00001.parquet|3           |

In [34]:
spark.sql("""
SELECT 
    partition,
    COUNT(*) AS qtd_arquivos,
    SUM(record_count) AS total_registros,
    SUM(file_size_in_bytes) AS tamanho_total
FROM hadoop_catalog.default.vendas_ecommerce.files
GROUP BY partition
""").show(truncate=False)

+-----------------+------------+---------------+-------------+
|partition        |qtd_arquivos|total_registros|tamanho_total|
+-----------------+------------+---------------+-------------+
|{54, Eletrônicos}|3           |4              |8549         |
|{53, Eletrônicos}|1           |3              |3023         |
|{54, Móveis}     |2           |2              |5037         |
|{53, Móveis}     |1           |2              |2293         |
+-----------------+------------+---------------+-------------+



In [35]:
spark.sql("""
SELECT 
    COUNT(*) AS arquivos_pequenos
FROM hadoop_catalog.default.vendas_ecommerce.files
WHERE file_size_in_bytes < 1000000
""").show()

+-----------------+
|arquivos_pequenos|
+-----------------+
|                7|
+-----------------+

